In [2]:
import requests
import re
import pandas as pd



df = pd.read_csv("../../02_Data/processed/final_gamefound_list.csv")

In [3]:
df

,name,projectID,creator,creatorID,homeUrl,currencySymbol,campaignGoal,fundsGathered,backersCount,campaignStart,...,minPlayers,maxPlayers,minAge,playTime,playTimeUnit,fundedInSeconds,imageUrl,pledgeManagerSoftCloseDeadline,projectTags,originalType
0,We Played God - A Survival Miniatures Game,5787,Black Site Studios,5252,https://gamefound.com/creators/black-site-studios,$,10000.0,393538.44,1567,2025-06-17T14:30:00Z,...,NaN,NaN,NaN,NaN,0,628,https://imgcdn.gamefound.com/projectimage/proj...,2026-06-02T15:00:00Z,"Exploration, Horror, Modern, Science Fiction, ...",1
1,CARNIVORE,6481,Buer Games,6737,https://gamefound.com/creators/buer-games,$,20000.0,247988.30,1075,2026-02-25T16:00:00Z,...,1.0,4.0,13.0,90.0,0,2716,https://imgcdn.gamefound.com/projectimage/proj...,2026-06-01T14:00:00Z,"History, Strategy, Wargame, Action, Collectibl...",1
2,1809: Talavera,8024,Tactical Workshop,5141,https://gamefound.com/creators/tactical-workshop,$,7000.0,23091.47,215,2026-03-01T05:00:00Z,...,1.0,2.0,12.0,8.0,1,56442,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"History, Strategy, Wargame",1
3,Comber Cat,8225,Gamedolf Limited,8739,https://gamefound.com/creators/gamedolflimited,$,300.0,645.66,12,2026-02-02T13:00:00Z,...,2.0,5.0,8.0,15.0,0,2409627,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Card Game, Strategy, Party game, Family",1
4,Jano - A Pawsome Adventure,5995,aszaki,7221,https://gamefound.com/creators/aszaki,€,1000.0,4042.67,89,2026-01-16T12:00:00Z,...,2.0,6.0,8.0,20.0,0,14439,https://imgcdn.gamefound.com/projectimage/proj...,2026-06-11T13:00:00Z,"Strategy, Resource management, Family, Worker ...",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,Tectonia,2109,Hagen Behr,2703,https://gamefound.com/creators/hagen-behr,€,20000.0,8001.00,113,2022-07-05T18:00:00Z,...,2.0,4.0,14.0,180.0,0,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Strategy, Wargame, Dice Game, Area Control, Co...",1
484,Invasion Cards,2149,Lawrence Lyle,2742,https://gamefound.com/creators/lawrence-lyle,€,9800.0,70.00,2,2022-07-04T10:00:00Z,...,2.0,4.0,5.0,25.0,0,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Fantasy, Strategy, Humor",1
485,MOOGH,1919,Nuts! Publishing,2363,https://gamefound.com/creators/nuts-publishing,€,10000.0,6285.00,143,2022-04-26T16:00:00Z,...,1.0,4.0,14.0,30.0,0,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Dice Game, Multiplayer, Competitive, Campaign",1
486,Crossroads Inn: The Board Game,1143,Klabater,1510,https://gamefound.com/creators/klabater,€,20000.0,13420.63,158,2021-10-28T14:00:00Z,...,2.0,4.0,12.0,120.0,0,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"Fantasy, Strategy, Terrain Building, Humor",1


In [5]:
import requests
import re
import pandas as pd
import time
from tqdm.notebook import tqdm


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

rewards_list= []
fail_list=[]
for idx, row in tqdm(df.iterrows(),total=len(df)):
    name = row['name']
    projectID_id = row['projectID']

    api_url = f"https://gamefound.com/api/projectContents/getRewards?projectID={projectID_id}"
    api_res = requests.get(api_url, headers=headers)
    if api_res.status_code == 200:
        data = api_res.json()
        if data['data']!=None:
            rewards_data = data.get('data').get('rewards')
        else:
            fail_list.append(api_url)            

        for item in rewards_data:
            item_dict = {
                'main_name' : name,
                'projectID_id' : projectID_id,
                "raw_url" : api_url,
                'anchorRelativeUrl': item.get('anchorRelativeUrl'),
                'backgroundUrl': item.get('backgroundUrl'),                # 상세페이지 대표 이미지
                'deliveryDateRemarks': item.get('deliveryDateRemarks'),
                'estimatedDeliveryAt': item.get('estimatedDeliveryAt'),    # 예상 배송일
                'hasDetails': item.get('hasDetails'),
                'isExposed': item.get('isExposed'),
                'isMostPopular': item.get('isMostPopular'),                # 가장 인기 있는 상품 여부
                'purchasedCopiesCount': item.get('purchasedCopiesCount'),  # 구매(후원)된 수량
                'additionalInfoUrl': item.get('additionalInfoUrl'),
                'hasInstallmentsAvailable': item.get('hasInstallmentsAvailable'), # 분할 납부 가능 여부
                'installmentCost': item.get('installmentCost'),
                'installmentMinPayment': item.get('installmentMinPayment'),
                'abstract': item.get('abstract'),                          # 상품 요약 설명
                'categoryID': item.get('categoryID'),
                'effectivePrice': item.get('effectivePrice'),              # 실제 결제 가격 (할인 적용 등)
                'hasLimitedStock': item.get('hasLimitedStock'),            # 한정 수량 여부
                'hasRetailerAccess': item.get('hasRetailerAccess'),
                'hasSpecialAccess': item.get('hasSpecialAccess'),
                'imageUrl': item.get('imageUrl'),                          # 상품 썸네일 이미지
                'isDigital': item.get('isDigital'),                        # 디지털 상품 여부
                'isDiscounted': item.get('isDiscounted'),                  # 할인 여부
                'isFeatured': item.get('isFeatured'),                      # 추천/강조 상품 여부
                'name': item.get('name'),                                  # 상품명
                'price': item.get('price'),                                # 원래 가격
                'productCanBePurchased': item.get('productCanBePurchased'), # 구매 가능 여부
                'productCanBePurchasedDescription': item.get('productCanBePurchasedDescription'),
                'productID': item.get('productID'),                        # 상품 고유 ID
                'projectID': item.get('projectID'),                        # 프로젝트 고유 ID
                'remainingStockLimit': item.get('remainingStockLimit'),    # 남은 수량
                'url': item.get('url'),                                    # 상품 상세 페이지 URL (상대 경로인 경우가 많음)
                'productState': item.get('productState')
            }
            rewards_list.append(item_dict)


    else:
        print(f"  {api_url} 오류 {api_res.status_code}")


time.sleep(1)

df1 = pd.DataFrame(rewards_list)

0it [00:00, ?it/s]

In [6]:
df1.to_csv("../../02_Data/raw/rewards.csv", index=False, encoding="utf-8-sig")